# N059 · K路归并与有序候选生成

**目标：** 维护每一路尚未消费的最小候选。

**先修：** N048, N057。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 堆头；惰性推进；元组比较；有序流；去重约定。

## 从问题到算法

多个有序流的全局最小值一定在某个流的当前头部。堆只需放每个流一个候选，取走后从同一流补充。两个有序数组的数对和可以把每个a[i]与整个b配对看成一条有序流。相同优先级要加可比较的索引，不能让Python继续比较节点对象。

## 最小实现

**本章接口：** `merge_k_lists(lists)`、`k_smallest_pairs(a, b, k)`

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = [], set()
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values

import heapq

def merge_k_lists(lists):
    heap=[(node.val,i,node) for i,node in enumerate(lists) if node]
    heapq.heapify(heap); dummy=tail=ListNode()
    while heap:
        _,i,node=heapq.heappop(heap)
        following=node.next
        tail.next=node; tail=node
        if following: heapq.heappush(heap,(following.val,i,following))
    if tail is not dummy: tail.next=None
    return dummy.next

def k_smallest_pairs(a,b,k):
    if not a or not b or k<=0: return []
    heap=[(a[i]+b[0],i,0) for i in range(min(k,len(a)))]; heapq.heapify(heap)
    out=[]
    while heap and len(out)<k:
        _,i,j=heapq.heappop(heap); out.append([a[i],b[j]])
        if j+1<len(b): heapq.heappush(heap,(a[i]+b[j+1],i,j+1))
    return out

## 正确性、前提与复杂度

每条流中未暴露元素不小于当前头部，因此堆顶是所有未输出候选中的最小值。只初始化前k行已经足够：更后行首不小于前k行首，无法成为严格更优的前k项。

**代价：** K路归并总N节点需O(N log K)时间/O(K)堆空间；数对输出q=min(k,mn)，时间O(q log min(k,m))。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,7,11]; b=[2,4,6]; pairs=k_smallest_pairs(a,b,6)
show_table(['弹出次序','数对','和'],[(i,p,sum(p)) for i,p in enumerate(pairs,1)])

弹出次序,数对,和
1,"[1, 2]",3
2,"[1, 4]",5
3,"[1, 6]",7
4,"[7, 2]",9
5,"[7, 4]",11
6,"[7, 6]",13


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
h=merge_k_lists([list_to_nodes([1,4,5]),None,list_to_nodes([1,3,4]),list_to_nodes([2,6])])
assert nodes_to_list(h)==[1,1,2,3,4,4,5,6]
assert merge_k_lists([]) is None
from itertools import combinations_with_replacement
for a in combinations_with_replacement([0,1,2],3):
    for b in combinations_with_replacement([0,1,2],2):
        all_sums=sorted(x+y for x in a for y in b)
        for k in range(8):
            out=k_smallest_pairs(a,b,k)
            assert [sum(p) for p in out]==all_sums[:k]
            assert all(x in a and y in b for x,y in out)
print('N059: 所有本章断言通过')

N059: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 实现多个有序生成器归并。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 避免比较不可排序节点对象。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 23. Merge k Sorted Lists（canonical）
- 373. Find K Pairs with Smallest Sums（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。